# DiploMind: filter articles for IR and current affairs

Runs **before** the dataset-generation notebook. It:
1. Loads every `.jsonl` file (1 Urdu file, 21 English files) and **removes duplicates** across files.
2. Cleans each article and drops ones that are too short.
3. Asks an LLM to classify each article: a one-line summary, a category, and `keep` / `drop` / `unsure`.
4. Writes a **report** to read through, and a **random sample** to check the filter's accuracy.
5. Exports only the kept articles, ready for the generation notebook.

Results are cached, so re-running costs nothing for articles already classified.

## 1. Install and API key

In [1]:
!pip install -q openai ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 901.8 kB/s eta 0:00:00 0:00:01


In [5]:
WANDB_API_KEY = ""   # paste your key here, or use a Kaggle secret named WANDB_API_KEY
WANDB_PROJECT = "m-ibrahim094131-fast-nuces/test"

import os
if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
print("API key set" if WANDB_API_KEY else "No API key yet: paste it above")

API key set


## 2. Settings

In [19]:
import os, re, csv, json, glob, time, random, hashlib, threading, collections
from concurrent.futures import ThreadPoolExecutor
from ftfy import fix_text
from openai import OpenAI

INPUT_DIRS = {
    "ur": "/kaggle/input/datasets/mibrahim094131/articles2",   # Urdu articles
    "en": "/kaggle/input/datasets/mibrahim094131/englih4",                     # folder with the 21 English .jsonl files
}
OUT_DIR = "filtered"            # on Kaggle this is /kaggle/working/filtered
MODEL   = "deepseek-ai/DeepSeek-V3.1"   # fast, no thinking; good enough for classification
WORKERS = 8
MIN_WORDS = 150                 # articles shorter than this after cleaning are dropped without an LLM call

os.makedirs(OUT_DIR, exist_ok=True)
CACHE_PATH = os.path.join(OUT_DIR, "filter_cache.jsonl")
client = OpenAI(base_url="https://api.inference.wandb.ai/v1", api_key=WANDB_API_KEY, project=WANDB_PROJECT)
print("Model:", MODEL, "| output:", OUT_DIR)

Model: deepseek-ai/DeepSeek-V3.1 | output: filtered


## 3. Load, clean and remove duplicates
The 21 English files are scrapes from different times, so the same article often appears in several of them. Duplicates are removed by link, then by title.

In [20]:
TAIL_MARKERS = ["Watch our pick of standout clips", "This breaking news story is being updated",
                "Get our flagship newsletter", "Copyright ©", "You can receive Breaking News"]

def clean_paragraphs(text):
    """Article paragraphs only: no page tail, captions, link lines or promo lines (same rules as the generation notebook)."""
    text = fix_text(text or "")
    for marker in TAIL_MARKERS:
        i = text.find(marker)
        if i > 0:
            text = text[:i]
    paras = []
    for p in text.split("\n"):
        p = p.strip()
        if len(p.split()) < 6:
            continue
        if not re.search(r"""[.!?۔؟:"”’‘')»]$""", p):
            continue
        if re.search(r"BBC|بی بی سی", p) and len(p.split()) < 15:
            continue
        paras.append(p)
    return paras

def norm_url(u):
    return re.sub(r"[?#].*$", "", (u or "").strip()).rstrip("/").lower()

def norm_title(t):
    return re.sub(r"\W+", " ", fix_text(t or "")).strip().lower()

raw, stats = [], collections.Counter()
for lang, folder in INPUT_DIRS.items():
    assert os.path.isdir(folder), f"Folder not found: {folder}"
    files = sorted(glob.glob(os.path.join(folder, "**", "*.jsonl"), recursive=True))
    print(f"{lang}: {len(files)} file(s)")
    for path in files:
        for n, line in enumerate(open(path, encoding="utf-8")):
            if not line.strip():
                continue
            d = json.loads(line)
            d["_lang"], d["_file"] = lang, os.path.basename(path)
            d["_url"] = d.get("url") or d.get("link") or f"{os.path.basename(path)}#{n}"
            d["_text"] = d.get("text") or d.get("content") or d.get("body") or ""
            raw.append(d)
            stats[f"loaded_{lang}"] += 1

articles, seen_url, seen_title = [], set(), set()
for d in raw:
    u, t = norm_url(d["_url"]), norm_title(d.get("title"))
    if u in seen_url or (t and t in seen_title):
        stats[f"duplicate_{d['_lang']}"] += 1
        continue
    seen_url.add(u)
    if t:
        seen_title.add(t)
    paras = clean_paragraphs(d["_text"])
    d["_words"] = sum(len(p.split()) for p in paras)
    d["_clean"] = "\n\n".join(paras)
    d["_id"] = hashlib.sha256(u.encode("utf-8")).hexdigest()[:16]
    if d["_words"] < MIN_WORDS:
        stats[f"too_short_{d['_lang']}"] += 1
        continue
    articles.append(d)

print(dict(stats))
print(len(articles), "unique articles to classify |", dict(collections.Counter(a["_lang"] for a in articles)))

ur: 1 file(s)
en: 24 file(s)
{'loaded_ur': 1200, 'loaded_en': 632, 'duplicate_en': 352, 'too_short_en': 37}
1443 unique articles to classify | {'ur': 1200, 'en': 243}


## 4. Classification prompt
The rules decide on the **main subject** of the article, not passing mentions. Domestic politics and governance counts as current affairs (for example the Khyber Pakhtunkhwa protests), while crime, sport and entertainment do not.

In [23]:
KEEP_CATS = ["diplomacy_foreign_policy", "conflict_security", "international_economy",
             "law_rights_state", "national_politics_governance", "migration_climate_policy"]
DROP_CATS = ["crime_individual", "sports", "entertainment_culture", "health_science_lifestyle",
             "business_technology", "accident_disaster", "human_interest", "other"]

SYS_FILTER = f"""You decide whether a news article belongs in a dataset that teaches international relations (IR) and current affairs to Pakistani readers.

KEEP the article if its MAIN subject is one of:
- diplomacy_foreign_policy: relations between countries, summits, treaties, visits, foreign policy decisions
- conflict_security: war, military operations, terrorism, armed groups, ceasefires, defence policy
- international_economy: trade, tariffs, sanctions, IMF or World Bank, debt, energy deals between countries
- law_rights_state: international law, human rights or courts where a state or international body is involved
- national_politics_governance: elections, governments, parliaments, political parties, protests against a government, constitutional or emergency measures, in Pakistan or any country
- migration_climate_policy: refugee, migration or climate policy decided by states or international bodies

DROP the article if its MAIN subject is one of:
- crime_individual: crimes, trials or police cases about private individuals with no political or state angle
- sports
- entertainment_culture: films, music, celebrities, arts, religion stories without a political angle
- health_science_lifestyle: health, science, education, lifestyle, unless it is mainly about state policy
- business_technology: companies, products, technology, unless it is mainly about trade policy or state regulation
- accident_disaster: accidents and natural disasters, unless it is mainly about government response or politics
- human_interest: personal stories, oddities
- other

Judge the main subject, not a passing mention of a country or politician.
Use "unsure" only if the article is truly half IR and half something else.

Reply with exactly one JSON object and nothing else:
{{"summary": "<one English sentence, at most 25 words, saying what the article is about>", "category": "<one of: {', '.join(KEEP_CATS + DROP_CATS)}>", "decision": "<keep | drop | unsure>", "reason": "<at most 12 words>"}}"""

def classify_messages(a):
    words = a["_clean"].split()
    body = " ".join(words[:300])
    return [{"role": "system", "content": SYS_FILTER},
            {"role": "user", "content": f"Title: {a.get('title', '')}\n\nArticle (start):\n{body}"}]

def parse_json(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S)
    m = re.search(r"\{.*\}", t, re.S)
    try:
        r = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        r = {}
    dec = str(r.get("decision", "")).strip().lower()
    cat = str(r.get("category", "")).strip().lower()
    return {"summary": str(r.get("summary", "")).strip(),
            "category": cat if cat in KEEP_CATS + DROP_CATS else "other",
            "decision": dec if dec in ("keep", "drop", "unsure") else "unsure",
            "reason": str(r.get("reason", "")).strip()}

print("Prompt ready")

Prompt ready


## 5. Classify (cached, parallel)

In [24]:
CACHE, LOCK = {}, threading.Lock()
if os.path.exists(CACHE_PATH):
    for line in open(CACHE_PATH, encoding="utf-8"):
        r = json.loads(line)
        CACHE[r["id"]] = r

def classify(a):
    if a["_id"] in CACHE:
        return CACHE[a["_id"]]
    for attempt in range(4):
        try:
            r = client.chat.completions.create(model=MODEL, max_tokens=200, temperature=0,
                                               messages=classify_messages(a))
            res = {"id": a["_id"], **parse_json(r.choices[0].message.content)}
            break
        except Exception as e:
            print(f"   [retry {attempt+1}] {type(e).__name__}: {str(e)[:80]}")
            time.sleep(2 ** attempt)
    else:
        return {"id": a["_id"], "summary": "", "category": "other", "decision": "unsure", "reason": "API error"}
    with LOCK:
        CACHE[a["_id"]] = res
        with open(CACHE_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")
    return res

t0 = time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    results = list(ex.map(classify, articles))
for a, r in zip(articles, results):
    a["_filter"] = r

# A keep decision with a drop category (or the reverse) is a contradiction: mark it unsure
for a in articles:
    f = a["_filter"]
    if (f["decision"] == "keep" and f["category"] in DROP_CATS) or (f["decision"] == "drop" and f["category"] in KEEP_CATS):
        f["decision"] = "unsure"
        f["reason"] = "category and decision disagree; " + f["reason"]

print(f"Done in {time.time() - t0:.0f}s")
for lang in INPUT_DIRS:
    c = collections.Counter(a["_filter"]["decision"] for a in articles if a["_lang"] == lang)
    print(f"{lang}: {dict(c)}")
print("Categories:", dict(collections.Counter(a["_filter"]["category"] for a in articles)))

Done in 11s
ur: {'keep': 633, 'drop': 566, 'unsure': 1}
en: {'keep': 213, 'drop': 29, 'unsure': 1}
Categories: {'national_politics_governance': 250, 'crime_individual': 268, 'conflict_security': 250, 'law_rights_state': 88, 'diplomacy_foreign_policy': 192, 'business_technology': 48, 'international_economy': 43, 'sports': 39, 'entertainment_culture': 106, 'health_science_lifestyle': 67, 'other': 13, 'human_interest': 47, 'migration_climate_policy': 25, 'accident_disaster': 7}


## 6. Report: read the decisions
Prints every article as `decision | category | title | summary | reason`, grouped by decision. `filter_report.csv` has the same, plus a `human` column you can fill in.

In [25]:
def show(group, limit=None):
    items = [a for a in articles if a["_filter"]["decision"] == group][:limit]
    print(f"\n{'=' * 30} {group.upper()} ({sum(a['_filter']['decision'] == group for a in articles)}) {'=' * 30}")
    for a in items:
        f = a["_filter"]
        print(f"[{a['_lang']}] {f['category']:<28} | {(a.get('title') or '')[:90]}")
        print(f"      {f['summary']}  ({f['reason']})")

show("unsure")
show("drop")
show("keep")

with open(os.path.join(OUT_DIR, "filter_report.csv"), "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "lang", "decision", "category", "title", "summary", "reason", "words", "url", "human"])
    for a in articles:
        fl = a["_filter"]
        w.writerow([a["_id"], a["_lang"], fl["decision"], fl["category"], a.get("title", ""),
                    fl["summary"], fl["reason"], a["_words"], a["_url"], ""])
print("\nSaved", os.path.join(OUT_DIR, "filter_report.csv"))


============================== UNSURE (2) ==============================
[ur] national_politics_governance | چاند پر جھنڈا: امریکہ کے بعد چین چاند پر جھنڈا گاڑنے والا دوسرا ملک بن گیا
      China plants its flag on the moon, becoming the second country after the US to do so, as part of a robotic lunar mission.  (category and decision disagree; Space achievement is national pride, not international relations)
[en] law_rights_state             | Christian metal band Demon Hunter sue Netflix over KPop Demon Hunters
      Christian metal band Demon Hunter sues Netflix over trademark infringement regarding the Kpop Demon Hunters film and merchandise.  (category and decision disagree; Private trademark dispute between company and media platform)

============================== DROP (595) ==============================
[ur] crime_individual             | ٹرین کی پٹری سے ملنے والی لاش کا معمہ اور خاتون کی آخری فون کال: ’ڈبے میں کچھ لوگ شراب پی
      A woman's mysterious death on a train in I

## 7. Check the filter's accuracy
Takes a **fixed random sample** of kept and dropped articles from each language, and numbers them.

1. Run the cell below and read each line.
2. Write the numbers of the **wrong** decisions in `WRONG` in the next cell.
3. Run the next cell: it prints how accurate each decision was.

The sample is fixed (`seed=7`), so the numbers stay the same if you re-run.

In [26]:
SAMPLE_PER_GROUP = 15   # per language, per decision (keep / drop)
rng = random.Random(7)
EVAL = []
for lang in INPUT_DIRS:
    for dec in ("keep", "drop"):
        pool = [a for a in articles if a["_lang"] == lang and a["_filter"]["decision"] == dec]
        EVAL += [(lang, dec, a) for a in rng.sample(pool, min(SAMPLE_PER_GROUP, len(pool)))]

for i, (lang, dec, a) in enumerate(EVAL, 1):
    f = a["_filter"]
    print(f"{i:>3}. [{lang}/{dec.upper():<4}] {(a.get('title') or '')[:85]}")
    print(f"       {f['summary']}")

  1. [ur/KEEP] تقسیم کے 70 برس: منشیات میں ڈوبے سرحدی دیہات
       Indian villagers near Pakistan border live in constant fear of conflict, with psychological impacts and landmine threats affecting daily life.
  2. [ur/KEEP] انڈیا میں سیاستدان ہاتھوں میں مچھلیاں اٹھا کر انتخابی مہم کیوں چلا رہے ہیں؟
       Indian politicians use fish in election campaigns to connect with Bengali cultural identity in West Bengal state elections.
  3. [ur/KEEP] پاکستان کی سرحدوں پر پابندیاں اور چمن کی تاریخ کا طویل احتجاج
       Pakistani workers protest prolonged border closure with Afghanistan at Chaman crossing, demanding restoration of livelihoods after COVID restrictions.
  4. [ur/KEEP] امریکہ سے 10 ارب ڈالر کے ایکسچینج سٹیبلائزیشن فیسیلٹی ملنے کی امید: کیا اس سے پاکستان
       Pakistan expects US approval of a $10 billion exchange stabilization facility to improve access to international capital markets.
  5. [ur/KEEP] لاہور میں انسٹاگرام سے شروع ہونے والے مبینہ ریپ کے غیرمصدقہ واقعے نے کیسے پُرتشد

In [27]:
WRONG = []   # e.g. [3, 17, 22]: numbers from the list above where the decision is wrong

wrong = set(WRONG)
print(f"{'group':<10} {'checked':>8} {'wrong':>6} {'correct':>8}")
for lang in INPUT_DIRS:
    for dec in ("keep", "drop"):
        idx = [i for i, (l, d, _) in enumerate(EVAL, 1) if l == lang and d == dec]
        if idx:
            bad = len(wrong & set(idx))
            print(f"{lang}/{dec:<6} {len(idx):>8} {bad:>6} {100 * (1 - bad / len(idx)):>7.0f}%")
print("\nkeep correct = precision (how clean the kept set is); drop correct = how few good articles are lost.")
print("Wrong decisions:")
for i in sorted(wrong):
    lang, dec, a = EVAL[i - 1]
    print(f"  {i}. [{lang}/{dec}] {a.get('title', '')[:90]}")

group       checked  wrong  correct
ur/keep         15      0     100%
ur/drop         15      0     100%
en/keep         15      0     100%
en/drop         15      0     100%

keep correct = precision (how clean the kept set is); drop correct = how few good articles are lost.
Wrong decisions:


## 8. Overrides, then export
- `UNSURE_POLICY`: what to do with `unsure` articles (`"drop"` is safer; `"keep"` gives more data). Read them in section 6 first.
- `FORCE_KEEP` / `FORCE_DROP`: titles (or parts of titles) you want to override by hand, for example the wrong ones from section 7.

Writes `filtered/ur/articles.jsonl` and `filtered/en/articles.jsonl` with the original fields, the cleaned text and the filter result. Point `ARTICLES_DIRS` in the generation notebook at these two folders.

In [29]:
UNSURE_POLICY = "drop"
FORCE_KEEP = []   # e.g. ["Liberia agrees to accept"]
FORCE_DROP = []   # e.g. ["Mushroom murderer"]

def final_keep(a):
    title = a.get("title") or ""
    if any(s.lower() in title.lower() for s in FORCE_DROP):
        return False
    if any(s.lower() in title.lower() for s in FORCE_KEEP):
        return True
    dec = a["_filter"]["decision"]
    return dec == "keep" or (dec == "unsure" and UNSURE_POLICY == "keep")

kept = collections.Counter()
for lang in INPUT_DIRS:
    os.makedirs(os.path.join(OUT_DIR, lang), exist_ok=True)
    with open(os.path.join(OUT_DIR, lang, "articles.jsonl"), "w", encoding="utf-8") as f:
        for a in articles:
            if a["_lang"] == lang and final_keep(a):
                out = {k: v for k, v in a.items() if not k.startswith("_")}
                out.update({"url": a["_url"], "text": a["_clean"], "lang": lang,
                            "ir_category": a["_filter"]["category"], "ir_summary": a["_filter"]["summary"]})
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
                kept[lang] += 1

print("Kept:", dict(kept), "| total:", sum(kept.values()))
print("Kept by category:", dict(collections.Counter(a["_filter"]["category"] for a in articles if final_keep(a))))

Kept: {'ur': 633, 'en': 213} | total: 846
Kept by category: {'national_politics_governance': 249, 'conflict_security': 250, 'law_rights_state': 87, 'diplomacy_foreign_policy': 192, 'international_economy': 43, 'migration_climate_policy': 25}
